# H3 Cascade — chạy lại bước prepare bị lỗi trên Ubuntu dual boot

Notebook này chạy **bước CPU đang lỗi ở full-pipeline.ipynb**: fit/reuse gate, calibration, routing, chuẩn bị Joern graph và prompt. Giữ cấu hình notebook đang lỗi: **ReVeal / test / baseline_whitening / full_reference=True**. Không có cell gọi LLM.

Đường dẫn theo notebook H2 Ubuntu:

- Project/kết quả H3: ~/.local/share/grace-h3/GRACE-H3-CalibratedCascade
- Joern: ~/.cache/grace-h2/joern-cli
- Java: ~/.cache/grace-h2/java-21
- Scratch: ~/.cache/grace-h3/tmp

Sau khi boot Ubuntu, mở terminal trong thư mục GRACE-H3-CalibratedCascade trên phân vùng Data, chạy một lần:

    python3 tools/migrate_ubuntu.py

Migration copy data, embeddings, graph cache, artifacts, runs và code sang native Ubuntu, bỏ qua Git, virtualenv, bundle ZIP và credential. Có thể chạy lại nếu copy bị ngắt; file Ubuntu khác nội dung được giữ và migration báo lỗi. Mở bản notebook trong thư mục Ubuntu sau khi migration hoàn tất. Không chạy hai notebook cùng ghi một project.

Nếu cần kernel riêng (cài đặt cần mạng):

    python3 -m venv ~/.venvs/grace-h3
    ~/.venvs/grace-h3/bin/python -m pip install -r ~/.local/share/grace-h3/GRACE-H3-CalibratedCascade/requirements-cpu.txt ipykernel
    ~/.venvs/grace-h3/bin/python -m ipykernel install --user --name grace-h3 --display-name "GRACE H3 Ubuntu"

Chọn kernel GRACE H3 Ubuntu. Có thể dùng kernel H2 Ubuntu nếu đủ dependency; restart riêng kernel notebook này để tránh import nhầm package từ project khác.


## U01 — Project native Ubuntu và cấu hình ReVeal


In [1]:
import os, sys, json, platform, time, tempfile, shutil, subprocess
from pathlib import Path

if platform.system() != 'Linux' or 'microsoft' in platform.release().lower():
    raise RuntimeError('Notebook dành cho Ubuntu dual boot, dùng kernel Linux native.')

LAB_ROOT = Path.home() / '.local/share/grace-h3'
ROOT = LAB_ROOT / 'GRACE-H3-CalibratedCascade'
if not (ROOT / 'grace_h3/workflow.py').is_file() or not (ROOT / 'data/manifest.json').is_file():
    raise FileNotFoundError(f'Chạy python3 tools/migrate_ubuntu.py từ bản H3 trên Data trước. Chưa đủ project/cache: {ROOT}')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['GRACE_H3_ROOT'] = str(ROOT)

JOERN_HOME_OVERRIDE = ''  # Optional absolute Linux path.
JOERN_HOME = Path(JOERN_HOME_OVERRIDE).expanduser() if JOERN_HOME_OVERRIDE else Path.home() / '.cache/grace-h2/joern-cli'
JAVA_HOME = Path.home() / '.cache/grace-h2/java-21'
LOCAL_TMP = Path.home() / '.cache/grace-h3/tmp'
LOCAL_TMP.mkdir(parents=True, exist_ok=True)
os.environ['JOERN_HOME'] = str(JOERN_HOME)
os.environ['TMPDIR'] = str(LOCAL_TMP)
tempfile.tempdir = str(LOCAL_TMP)
if (JAVA_HOME / 'bin/java').is_file():
    os.environ['JAVA_HOME'] = str(JAVA_HOME)
    os.environ['PATH'] = str(JAVA_HOME / 'bin') + os.pathsep + os.environ['PATH']

config = json.loads((ROOT / 'configs/experiment.json').read_text(encoding='utf-8'))
config.update(datasets=['reveal'], split='test', full_reference=True, retrieval_arm='baseline_whitening')
config['graph_workers'] = 4
print(json.dumps({'root': str(ROOT), 'joern': str(JOERN_HOME), 'scratch': str(LOCAL_TMP), 'config': config}, indent=2))


{
  "root": "/home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade",
  "joern": "/home/quang-dung/.cache/grace-h2/joern-cli",
  "scratch": "/home/quang-dung/.cache/grace-h3/tmp",
  "config": {
    "datasets": [
      "reveal"
    ],
    "split": "test",
    "full_reference": true,
    "retrieval_arm": "baseline_whitening",
    "graph_workers": 4,
    "inference_workers": 4,
    "bootstrap": 1000,
    "gate": {
      "C": 1,
      "class_weight": null,
      "max_iter": 1000,
      "seed": 42,
      "calibration_fraction": 0.5,
      "low_grid": [
        0.01,
        0.025,
        0.05,
        0.075,
        0.1,
        0.15,
        0.2,
        0.25,
        0.3
      ],
      "high_grid": [
        0.7,
        0.8,
        0.9,
        0.95,
        0.975,
        0.99
      ],
      "enable_direct_high": false,
      "max_lost_positive_fraction": 0.01,
      "min_high_precision": 0.98,
      "max_high_false_positive_fraction": 0.01,
      "family_alpha": 0.05
    }

## U02 — Kiểm tra dependency, cache và Joern Linux

Cell không cài package/tải runtime. Nếu runtime H2 nằm chỗ khác, sửa JOERN_HOME_OVERRIDE ở U01.


In [2]:
import importlib.util
required = ['numpy', 'scipy', 'sklearn', 'yaml', 'rapidfuzz', 'dotenv', 'pandas']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'Thiếu dependency trong kernel: {missing}; xem lệnh cài ở đầu notebook.')
for name in ('joern-parse', 'joern-export'):
    launcher = JOERN_HOME / name
    if not launcher.is_file() or not os.access(launcher, os.X_OK):
        raise RuntimeError(f'Joern launcher thiếu hoặc không executable: {launcher}')
java = shutil.which('java')
if not java:
    raise RuntimeError('Chưa có Java trong kernel Ubuntu; dùng Java 21 giống H2.')
info = subprocess.run([java, '-version'], capture_output=True, text=True, timeout=30)
print((info.stderr or info.stdout).strip(), flush=True)
if info.returncode: raise RuntimeError('Java không khởi động được.')
startup = subprocess.run([str(JOERN_HOME / 'joern-parse'), '-J-Xms128m', '-J-Xmx2g', '--help'],
                         capture_output=True, text=True, timeout=120)
if startup.returncode:
    raise RuntimeError('Joern startup thất bại: ' + (startup.stderr + startup.stdout)[-3000:])
manifest = json.loads((ROOT / 'data/manifest.json').read_text(encoding='utf-8'))
for dataset in config['datasets']:
    for split in ('train', 'validation', config['split']):
        item = manifest['datasets'][dataset][split]
        paths = [ROOT / item['rows_path']]
        paths += [ROOT / 'cache/embeddings' / dataset / split / name for name in ('manifest.json', 'vectors.npy', 'index.jsonl')]
        absent = [str(path) for path in paths if not path.is_file()]
        if absent: raise FileNotFoundError('Thiếu data/embedding cache: ' + str(absent))
print('Joern startup và các file cache cần thiết đã sẵn sàng.', flush=True)


openjdk version "21.0.12.1" 2026-08-18 LTS
OpenJDK Runtime Environment Temurin-21.0.12.1+1 (build 21.0.12.1+1-LTS)
OpenJDK 64-Bit Server VM Temurin-21.0.12.1+1 (build 21.0.12.1+1-LTS, mixed mode, sharing)
Joern startup và các file cache cần thiết đã sẵn sàng.


## U03 — Thử Joern parse/export cho một hàm C nhỏ

Kiểm tra runtime thực tế trước bước prepare dài. Graph thử ở runs/ubuntu_prepare_smoke, không dùng mẫu test hay nhãn. Nếu export vẫn lỗi, mở last-export-diagnostics.json theo exception để xem danh sách file và log Joern.


In [3]:
from grace_h1.graphs import GraphStore
from grace_h1.io import sha256

smoke_code = 'int grace_ubuntu_smoke(int x) { if (x > 0) return x; return 0; }'
row = {'record_id': 'runtime-smoke', 'code': smoke_code, 'code_sha256': sha256(smoke_code)}
smoke_store = GraphStore(ROOT / 'runs/ubuntu_prepare_smoke', joern_home=str(JOERN_HOME))
smoke_graph = smoke_store.ensure(row)
if not smoke_graph['nodes'] or not smoke_graph['edges']:
    raise RuntimeError('Joern smoke tạo graph rỗng; xử lý runtime trước khi prepare toàn bộ.')
print({'smoke_nodes': len(smoke_graph['nodes']), 'smoke_edges': len(smoke_graph['edges']), 'joern_home': smoke_store.home})


{'smoke_nodes': 14, 'smoke_edges': 80, 'joern_home': '/home/quang-dung/.cache/grace-h2/joern-cli'}


## U04 — Resume đúng bước CPU đang lỗi

Pipeline kiểm tra gate/embedding/graph theo fingerprint. Graph hoàn tất được lưu trên đĩa; rerun dùng lại cache phù hợp. Nếu sửa code/runtime, artifact đóng băng không khớp có thể cần tạo lại; không tự đổi hash để ép reuse.

Heartbeat in mỗi 30 giây. Nếu dừng, chờ cell và Joern worker kết thúc rồi mới chạy lại; không chạy hai phiên prepare đồng thời.


In [4]:
import threading
from grace_h3.workflow import prepare_all, selected_jobs
from grace_h1.io import write_json

stop_heartbeat = threading.Event()
started = time.monotonic()
def heartbeat():
    while not stop_heartbeat.wait(30):
        print(f'CPU prepare vẫn chạy: {(time.monotonic()-started)/60:.1f} phút; graph hoàn tất lưu trực tiếp trong cache/graphs.', flush=True)
def checkpoint():
    print('CPU checkpoint: graph/job hoàn tất nằm trên đĩa; không đóng gói lại cache.', flush=True)
worker = threading.Thread(target=heartbeat, daemon=True)
worker.start()
try:
    prepared = prepare_all(ROOT, config, checkpoint_callback=checkpoint)
    jobs = selected_jobs(ROOT, config)
    write_json(ROOT / 'runs/ubuntu_prepare_completed.json',
               {'config': config, 'runs': prepared, 'elapsed_seconds': time.monotonic()-started,
                'root': str(ROOT), 'joern_home': str(JOERN_HOME), 'llm_called': False})
finally:
    stop_heartbeat.set()
    worker.join(timeout=2)
print('Prepare hoàn tất:', prepared, flush=True)


/home/quang-dung/.venvs/grace-h2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


reveal/baseline_whitening: 9819 unique graph features required


reveal graphs:   0%|          | 0/9819 [00:00<?, ?graph/s]

CPU checkpoint: graph/job hoàn tất nằm trên đĩa; không đóng gói lại cache.


reveal graphs:  21%|██▏       | 2102/9819 [00:11<00:52, 145.77graph/s, built=0, cached=2102, new_per_min=0.00]

...

reveal graphs: 100%|█████████▉| 9806/9819 [2:20:42<00:32,  2.50s/graph, built=3057, cached=6749, new_per_min=21.73]

CPU prepare vẫn chạy: 141.0 phút; graph hoàn tất lưu trực tiếp trong cache/graphs.


reveal graphs: 100%|██████████| 9819/9819 [2:21:11<00:00,  1.16graph/s, built=3070, cached=6749, new_per_min=21.74]

Graph stage: 3070 new, 6749 cached; 141.20 min; 21.74 new graphs/min (4 workers)


CPU prepare vẫn chạy: 141.5 phút; graph hoàn tất lưu trực tiếp trong cache/graphs.
prepared 2274 prompts; graph coverage 95.1%: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/runs/reveal/test/baseline_whitening/8d73c4d11b68d215
CPU checkpoint: graph/job hoàn tất nằm trên đĩa; không đóng gói lại cache.
reuse frozen preparation: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/runs/reveal/test/baseline_whitening/8d73c4d11b68d215
CPU checkpoint: graph/job hoàn tất nằm trên đĩa; không đóng gói lại cache.
Prepare hoàn tất: [{'dataset': 'reveal', 'path': 'runs/h3/reveal/c3667b15c2909783'}]


## U05 — Xem policy và tiếp tục full pipeline tại bản Ubuntu

Mở ROOT/notebooks/full-pipeline.ipynb trên Ubuntu, dùng cùng ROOT/config. Chạy cell setup rồi tiếp tục từ bước đọc policy/preflight; selection nằm ở runs/h3_selection.json trong project native. Nếu chạy lại prepare, pipeline kiểm tra/reuse job đóng băng phù hợp.

Full-pipeline đang đặt RUN_INFERENCE=True; chỉ chạy cell inference khi sẵn sàng gọi API. Notebook prepare không thay credential, import bundle cũ hoặc xuất ZIP. Kết quả Ubuntu chưa tự đồng bộ về bản Data/Windows.


In [5]:
import pandas as pd
from grace_h1.io import read_json
table = []
for folder, job in jobs:
    gate = read_json(folder / 'gate_manifest.json')
    table.append({'dataset': job['dataset'], 'n': job['n'], 'branches': job['branch_counts'],
                  'low': gate['policy']['low'], 'high': gate['policy']['high'],
                  'inspect_path': job['inspect_path'], 'reference_path': job['reference_path']})
display(pd.DataFrame(table))
print('Selection:', ROOT / 'runs/h3_selection.json')
print('Notebook tiếp tục:', ROOT / 'notebooks/full-pipeline.ipynb')


,dataset,n,branches,low,high,inspect_path,reference_path
0,reveal,2274,{'inspect': 2274},0.0,None,runs/reveal/test/baseline_whitening/8d73c4d11b...,runs/reveal/test/baseline_whitening/8d73c4d11b...


Selection: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/runs/h3_selection.json
Notebook tiếp tục: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/notebooks/full-pipeline.ipynb


In [7]:
# U06 — Retry empty graphs in an isolated, versioned cache (run this cell manually).
import concurrent.futures as cf
import gzip, hashlib, json, os, subprocess, time
from pathlib import Path
from tqdm.auto import tqdm
from grace_h1.reference import parse_graphml
from grace_h1.io import fingerprint

retry_root = ROOT / 'runs' / 'graph_retry_v1'
retry_root.mkdir(parents=True, exist_ok=True)
# These annotation macros expand to nothing; semantic macros are left untouched.
annotation_macros = ('_U_', 'ATTRIBUTE_UNUSED', 'HB_UNUSED', 'ATTR_UNUSED')
retry_profile = {'version': 1, 'annotation_macros': list(annotation_macros),
                 'language_extension': '.cpp', 'heap': '2g', 'processors': 4,
                 'joern_home': str(JOERN_HOME)}
# Include Joern launcher content in the profile; outputs also retain subprocess logs.
retry_profile['parse_launcher_sha256'] = hashlib.sha256((JOERN_HOME / 'joern-parse').read_bytes()).hexdigest()
profile_id = fingerprint(retry_profile)[:16]
retry_dir = retry_root / profile_id
retry_dir.mkdir(exist_ok=True)
(retry_dir / 'profile.json').write_text(json.dumps(retry_profile, indent=2))

# Use the selected run's complete graph index: targets AND retrieval candidates.
selection = json.loads((ROOT / 'runs/h3_selection.json').read_text())
hashes = set()
for selected in selection['runs']:
    job = json.loads((ROOT / selected['path'] / 'job.json').read_text())
    for key in ('inspect_path', 'reference_path'):
        if job.get(key):
            for line in (ROOT / job[key] / 'graph_feature_index.jsonl').read_text().splitlines():
                entry = json.loads(line)
                if not entry['nodes'] or not entry['edges']:
                    hashes.add(entry['code_sha256'])
rows_by_hash = {}
manifest = json.loads((ROOT / 'data/manifest.json').read_text())
for dataset in config['datasets']:
    for split, item in manifest['datasets'][dataset].items():
        if not isinstance(item, dict) or 'rows_path' not in item:
            continue
        with (ROOT / item['rows_path']).open() as handle:
            for line in handle:
                row = json.loads(line)
                if row['code_sha256'] in hashes:
                    rows_by_hash.setdefault(row['code_sha256'], row)
missing_rows = hashes - rows_by_hash.keys()
if missing_rows:
    raise RuntimeError(f'Cannot locate source for {len(missing_rows)} empty graphs')
print(f'Retrying {len(hashes)} empty graphs; isolated output: {retry_dir}', flush=True)

# Restore split :: only outside quoted strings and comments.
def restore_scope_tokens(code):
    import re
    token = re.compile(r'R"(?P<delimiter>[^ ()\\\t\r\n]{0,16})\(.*?\)(?P=delimiter)"|"(?:\\.|[^"\\])*"|\'(?:\\.|[^\'\\])*\'|//[^\n]*|/\*.*?\*/|:\s+:', re.S)
    return token.sub(lambda m: '::' if m.group(0).startswith(':') else m.group(0), code)

def retry_one(code_hash):
    row = rows_by_hash[code_hash]
    card = retry_dir / code_hash
    card.mkdir(exist_ok=True)
    result_path = card / 'result.json'
    if result_path.exists():
        previous = json.loads(result_path.read_text())
        if previous.get('status') == 'recovered' and (card / 'features.json.gz').is_file():
            return previous
    original = row['code']
    if hashlib.sha256(original.encode('utf-8')).hexdigest() != code_hash:
        raise ValueError(f'Source identity mismatch: {code_hash}')
    # Reject cases needing genuine macro expansion/build context, rather than invent it.
    if '##' in original:
        result = {'code_sha256': code_hash, 'record_id': row['record_id'],
                  'status': 'needs_macro_context', 'nodes': 0, 'edges': 0}
        result_path.write_text(json.dumps(result, indent=2))
        return result
    prefix = ''.join(f'#ifndef {name}\n#define {name}\n#endif\n' for name in annotation_macros)
    parser_input = prefix + restore_scope_tokens(original)
    source = card / 'target.cpp'
    source.write_text(parser_input, encoding='utf-8')
    result = {'code_sha256': code_hash, 'record_id': row['record_id'],
              'parser_input_sha256': hashlib.sha256(parser_input.encode()).hexdigest(),
              'profile_id': profile_id, 'status': 'failed', 'nodes': 0, 'edges': 0}
    started = time.monotonic()
    try:
        import shutil
        export = card / 'export'
        if export.exists():
            shutil.rmtree(export)
        commands = [
            ('joern-parse', [str(source), '--output', str(card / 'cpg.bin'), '--frontend-args', '--log-problems']),
            ('joern-export', ['--repr', 'all', '--format', 'graphml', '--out', str(export), str(card / 'cpg.bin')])]
        for tool, args in commands:
            completed = subprocess.run([str(JOERN_HOME / tool), '-J-Xms128m', '-J-Xmx2g',
                                        '-J-XX:ActiveProcessorCount=4', *args],
                                       capture_output=True, text=True, errors='replace', timeout=300)
            (card / (tool + '.log')).write_text(completed.stdout + '\n' + completed.stderr)
            if completed.returncode:
                raise RuntimeError(f'{tool} exited {completed.returncode}; see {card / (tool + ".log")}')
        files = [p for p in export.rglob('*') if p.suffix.lower() in ('.xml', '.graphml')]
        if len(files) != 1:
            raise RuntimeError(f'Expected 1 GraphML, found {len(files)}')
        graph = parse_graphml(files[0])
        result.update(nodes=len(graph['nodes']), edges=len(graph['edges']))
        result['status'] = 'recovered' if graph['nodes'] and graph['edges'] else 'still_empty'
        graph.update(schema='grace_h1_graph_v1', code_sha256=code_hash,
                     source='joern_retry_derived_input_v1', retry_provenance=dict(result))
        graph['feature_sha256'] = fingerprint(graph)
        with gzip.open(card / 'features.tmp.gz', 'wt', encoding='utf-8') as handle:
            json.dump(graph, handle, ensure_ascii=False, sort_keys=True)
        os.replace(card / 'features.tmp.gz', card / 'features.json.gz')
    except Exception as exc:
        result['error'] = str(exc)
    result['elapsed_seconds'] = time.monotonic() - started
    temp = card / 'result.tmp.json'
    temp.write_text(json.dumps(result, ensure_ascii=False, indent=2))
    temp.replace(result_path)
    return result

results = []
started = time.monotonic()
workers = min(4, int(config.get('graph_workers', 4)))
with cf.ThreadPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(retry_one, h) for h in sorted(hashes)]
    with tqdm(total=len(futures), desc='Retry empty graphs', unit='graph', mininterval=1) as bar:
        for future in cf.as_completed(futures):
            results.append(future.result())
            from collections import Counter
            counts = Counter(item['status'] for item in results)
            bar.set_postfix(dict(counts), refresh=False)
            bar.update(1)
results.sort(key=lambda item: item['code_sha256'])
report = {'profile': retry_profile, 'profile_id': profile_id, 'total': len(hashes),
          'counts': dict(Counter(item['status'] for item in results)),
          'elapsed_seconds': time.monotonic() - started, 'results': results,
          'production_cache_updated': False, 'prompts_regenerated': False}
report_path = retry_dir / 'report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps({k: v for k, v in report.items() if k not in ('results', 'profile')}, indent=2))
print('Report:', report_path)
print('Recovered graphs are isolated. Existing prompts/frozen runs still use the original cache.')


Retrying 805 empty graphs; isolated output: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/runs/graph_retry_v1/4f0531a2faaa363d


Retry empty graphs: 100%|██████████| 805/805 [06:21<00:00,  2.11graph/s, recovered=590, needs_macro_context=59, still_empty=156]

{
  "profile_id": "4f0531a2faaa363d",
  "total": 805,
  "counts": {
    "still_empty": 156,
    "recovered": 590,
    "needs_macro_context": 59
  },
  "elapsed_seconds": 381.64614368299954,
  "production_cache_updated": false,
  "prompts_regenerated": false
}
Report: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/runs/graph_retry_v1/4f0531a2faaa363d/report.json
Recovered graphs are isolated. Existing prompts/frozen runs still use the original cache.


In [8]:
# U07 — Retry v2: additional parser annotation context; preserve original code/cache.
import concurrent.futures as cf
import gzip, hashlib, json, os, subprocess, time
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
import re
from grace_h1.reference import parse_graphml
from grace_h1.io import fingerprint

retry_root = ROOT / 'runs' / 'graph_retry_v2'
retry_root.mkdir(parents=True, exist_ok=True)
# These annotation macros expand to nothing; semantic macros are left untouched.
annotation_macros = ('_U_', 'ATTRIBUTE_UNUSED', 'HB_UNUSED', 'ATTR_UNUSED',
                     'U_CALLCONV', 'U_EXPORT2', '__user', '__always_inline',
                     'magick_restrict', 'PNGAPI', 'av_unused')
macro_values = {name: '' for name in annotation_macros}
macro_values['__always_inline'] = 'inline'
retry_profile = {'version': 2, 'annotation_macros': macro_values,
                 'interpretation': 'parser annotation abstraction; not full project preprocessing',
                 'language_extension': '.cpp', 'heap': '2g', 'processors': 4,
                 'joern_home': str(JOERN_HOME)}
# Include Joern launcher content in the profile; outputs also retain subprocess logs.
retry_profile['parse_launcher_sha256'] = hashlib.sha256((JOERN_HOME / 'joern-parse').read_bytes()).hexdigest()
profile_id = fingerprint(retry_profile)[:16]
retry_dir = retry_root / profile_id
retry_dir.mkdir(exist_ok=True)
(retry_dir / 'profile.json').write_text(json.dumps(retry_profile, indent=2))

# Use the selected run's complete graph index: targets AND retrieval candidates.
selection = json.loads((ROOT / 'runs/h3_selection.json').read_text())
hashes = set()
for selected in selection['runs']:
    job = json.loads((ROOT / selected['path'] / 'job.json').read_text())
    for key in ('inspect_path', 'reference_path'):
        if job.get(key):
            for line in (ROOT / job[key] / 'graph_feature_index.jsonl').read_text().splitlines():
                entry = json.loads(line)
                if not entry['nodes'] or not entry['edges']:
                    hashes.add(entry['code_sha256'])
# Preserve all successful v1 outputs; only retry the remaining entries.
previous_reports = list((ROOT / 'runs/graph_retry_v1').glob('*/report.json'))
if not previous_reports:
    raise RuntimeError('Run U06 first: no graph_retry_v1 report found.')
recovered_v1 = set()
for previous_path in previous_reports:
    previous = json.loads(previous_path.read_text())
    for item in previous['results']:
        if (item['status'] == 'recovered' and
                (previous_path.parent / item['code_sha256'] / 'features.json.gz').is_file()):
            recovered_v1.add(item['code_sha256'])
hashes -= recovered_v1
print(f'Skipping {len(recovered_v1)} recovered v1 graphs; {len(hashes)} unresolved.', flush=True)
rows_by_hash = {}
manifest = json.loads((ROOT / 'data/manifest.json').read_text())
for dataset in config['datasets']:
    for split, item in manifest['datasets'][dataset].items():
        if not isinstance(item, dict) or 'rows_path' not in item:
            continue
        with (ROOT / item['rows_path']).open() as handle:
            for line in handle:
                row = json.loads(line)
                if row['code_sha256'] in hashes:
                    rows_by_hash.setdefault(row['code_sha256'], row)
missing_rows = hashes - rows_by_hash.keys()
if missing_rows:
    raise RuntimeError(f'Cannot locate source for {len(missing_rows)} empty graphs')
print(f'Retrying {len(hashes)} empty graphs; isolated output: {retry_dir}', flush=True)

# Restore split :: only outside quoted strings and comments.
def restore_scope_tokens(code):
    import re
    token = re.compile(r'R"(?P<delimiter>[^ ()\\\t\r\n]{0,16})\(.*?\)(?P=delimiter)"|"(?:\\.|[^"\\])*"|\'(?:\\.|[^\'\\])*\'|//[^\n]*|/\*.*?\*/|:\s+:', re.S)
    return token.sub(lambda m: '::' if m.group(0).startswith(':') else m.group(0), code)

def retry_one(code_hash):
    row = rows_by_hash[code_hash]
    card = retry_dir / code_hash
    card.mkdir(exist_ok=True)
    result_path = card / 'result.json'
    if result_path.exists():
        previous = json.loads(result_path.read_text())
        if previous.get('status') == 'recovered' and (card / 'features.json.gz').is_file():
            return previous
    original = row['code']
    if hashlib.sha256(original.encode('utf-8')).hexdigest() != code_hash:
        raise ValueError(f'Source identity mismatch: {code_hash}')
    # Reject cases needing genuine macro expansion/build context, rather than invent it.
    # Thread context macros can add parameters/arguments: never erase them blindly.
    if re.search(r'\b(?:TSRMLS_[A-Z]+|EV_P_|EV_A_)\b', original):
        result = {'code_sha256': code_hash, 'record_id': row['record_id'],
                  'status': 'needs_project_context', 'nodes': 0, 'edges': 0,
                  'reason': 'thread/event context macro requires original build definitions'}
        result_path.write_text(json.dumps(result, indent=2))
        return result
    if '##' in original:
        result = {'code_sha256': code_hash, 'record_id': row['record_id'],
                  'status': 'needs_macro_context', 'nodes': 0, 'edges': 0}
        result_path.write_text(json.dumps(result, indent=2))
        return result
    active_macros = [name for name in annotation_macros
                     if re.search(r'\b' + re.escape(name) + r'\b', original)]
    if not active_macros:
        result = {'code_sha256': code_hash, 'record_id': row['record_id'],
                  'status': 'needs_project_context', 'nodes': 0, 'edges': 0,
                  'reason': 'no additional supported annotation; requires individual source/build investigation'}
        result_path.write_text(json.dumps(result, indent=2))
        return result
    prefix = ''.join(f'#ifndef {name}\n#define {name} {macro_values[name]}\n#endif\n'
                     for name in active_macros)
    parser_input = prefix + restore_scope_tokens(original)
    source = card / 'target.cpp'
    source.write_text(parser_input, encoding='utf-8')
    result = {'code_sha256': code_hash, 'record_id': row['record_id'],
              'parser_input_sha256': hashlib.sha256(parser_input.encode()).hexdigest(),
              'profile_id': profile_id, 'applied_annotations': active_macros, 'status': 'failed', 'nodes': 0, 'edges': 0}
    started = time.monotonic()
    try:
        import shutil
        export = card / 'export'
        if export.exists():
            shutil.rmtree(export)
        commands = [
            ('joern-parse', [str(source), '--output', str(card / 'cpg.bin'), '--frontend-args', '--log-problems']),
            ('joern-export', ['--repr', 'all', '--format', 'graphml', '--out', str(export), str(card / 'cpg.bin')])]
        for tool, args in commands:
            completed = subprocess.run([str(JOERN_HOME / tool), '-J-Xms128m', '-J-Xmx2g',
                                        '-J-XX:ActiveProcessorCount=4', *args],
                                       capture_output=True, text=True, errors='replace', timeout=300)
            (card / (tool + '.log')).write_text(completed.stdout + '\n' + completed.stderr)
            if completed.returncode:
                raise RuntimeError(f'{tool} exited {completed.returncode}; see {card / (tool + ".log")}')
        files = [p for p in export.rglob('*') if p.suffix.lower() in ('.xml', '.graphml')]
        if len(files) != 1:
            raise RuntimeError(f'Expected 1 GraphML, found {len(files)}')
        graph = parse_graphml(files[0])
        result.update(nodes=len(graph['nodes']), edges=len(graph['edges']))
        result['status'] = 'recovered' if graph['nodes'] and graph['edges'] else 'still_empty'
        graph.update(schema='grace_h1_graph_v1', code_sha256=code_hash,
                     source='joern_retry_derived_input_v2', retry_provenance=dict(result))
        graph['feature_sha256'] = fingerprint(graph)
        with gzip.open(card / 'features.tmp.gz', 'wt', encoding='utf-8') as handle:
            json.dump(graph, handle, ensure_ascii=False, sort_keys=True)
        os.replace(card / 'features.tmp.gz', card / 'features.json.gz')
    except Exception as exc:
        result['error'] = str(exc)
    result['elapsed_seconds'] = time.monotonic() - started
    temp = card / 'result.tmp.json'
    temp.write_text(json.dumps(result, ensure_ascii=False, indent=2))
    temp.replace(result_path)
    return result

results = []
started = time.monotonic()
workers = min(4, int(config.get('graph_workers', 4)))
with cf.ThreadPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(retry_one, h) for h in sorted(hashes)]
    with tqdm(total=len(futures), desc='Retry empty graphs', unit='graph', mininterval=1) as bar:
        for future in cf.as_completed(futures):
            results.append(future.result())
            from collections import Counter
            counts = Counter(item['status'] for item in results)
            bar.set_postfix(dict(counts), refresh=False)
            bar.update(1)
results.sort(key=lambda item: item['code_sha256'])
report = {'profile': retry_profile, 'profile_id': profile_id, 'total': len(hashes),
          'counts': dict(Counter(item['status'] for item in results)),
          'elapsed_seconds': time.monotonic() - started, 'results': results,
          'production_cache_updated': False, 'prompts_regenerated': False,
          'v1_recovered_preserved': len(recovered_v1)}
report_path = retry_dir / 'report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps({k: v for k, v in report.items() if k not in ('results', 'profile')}, indent=2))
print('Report:', report_path)
print('Recovered graphs are isolated. Existing prompts/frozen runs still use the original cache.')


Skipping 590 recovered v1 graphs; 215 unresolved.
Retrying 215 empty graphs; isolated output: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/runs/graph_retry_v2/0f19bff98c6877a8


Retry empty graphs: 100%|██████████| 215/215 [04:04<00:00,  1.14s/graph, needs_project_context=69, needs_macro_context=59, recovered=87]

{
  "profile_id": "0f19bff98c6877a8",
  "total": 215,
  "counts": {
    "recovered": 87,
    "needs_macro_context": 59,
    "needs_project_context": 69
  },
  "elapsed_seconds": 244.3388268869985,
  "production_cache_updated": false,
  "prompts_regenerated": false,
  "v1_recovered_preserved": 590
}
Report: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/runs/graph_retry_v2/0f19bff98c6877a8/report.json
Recovered graphs are isolated. Existing prompts/frozen runs still use the original cache.


In [9]:
# U08 — Retry v3: explicit project profiles; never fabricate missing functions.
import concurrent.futures as cf
import gzip, hashlib, json, os, subprocess, time
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
import re
from grace_h1.reference import parse_graphml
from grace_h1.io import fingerprint

retry_root = ROOT / 'runs' / 'graph_retry_v3'
retry_root.mkdir(parents=True, exist_ok=True)
# These annotation macros expand to nothing; semantic macros are left untouched.
annotation_macros = ('_U_', 'ATTRIBUTE_UNUSED', 'HB_UNUSED', 'ATTR_UNUSED',
                     'U_CALLCONV', 'U_EXPORT2', '__user', '__always_inline',
                     'magick_restrict', 'PNGAPI', 'av_unused')
macro_values = {name: '' for name in annotation_macros}
macro_values['__always_inline'] = 'inline'
retry_profile = {'version': 3, 'annotation_macros': macro_values,
                 'interpretation': 'parser annotation abstraction; not full project preprocessing',
                 'language_extension': '.cpp', 'heap': '2g', 'processors': 4,
                 'joern_home': str(JOERN_HOME)}
# Include Joern launcher content in the profile; outputs also retain subprocess logs.
retry_profile['php_configuration'] = 'NTS hypothesis; original build configuration unknown'
retry_profile['additional_profiles'] = ['PHP_NTS', 'OpenSSL_STACK_OF', 'reg1_OUT_LDAP_CONST']
retry_profile['parse_launcher_sha256'] = hashlib.sha256((JOERN_HOME / 'joern-parse').read_bytes()).hexdigest()
profile_id = fingerprint(retry_profile)[:16]
retry_dir = retry_root / profile_id
retry_dir.mkdir(exist_ok=True)
(retry_dir / 'profile.json').write_text(json.dumps(retry_profile, indent=2))

# Use the selected run's complete graph index: targets AND retrieval candidates.
selection = json.loads((ROOT / 'runs/h3_selection.json').read_text())
hashes = set()
for selected in selection['runs']:
    job = json.loads((ROOT / selected['path'] / 'job.json').read_text())
    for key in ('inspect_path', 'reference_path'):
        if job.get(key):
            for line in (ROOT / job[key] / 'graph_feature_index.jsonl').read_text().splitlines():
                entry = json.loads(line)
                if not entry['nodes'] or not entry['edges']:
                    hashes.add(entry['code_sha256'])
# Preserve all successful v1 outputs; only retry the remaining entries.
previous_reports = [p for version in ('graph_retry_v1', 'graph_retry_v2')
                    for p in (ROOT / 'runs' / version).glob('*/report.json')]
if not previous_reports:
    raise RuntimeError('Run U06 first: no graph_retry_v1 report found.')
recovered_v1 = set()
for previous_path in previous_reports:
    previous = json.loads(previous_path.read_text())
    for item in previous['results']:
        if (item['status'] == 'recovered' and
                (previous_path.parent / item['code_sha256'] / 'features.json.gz').is_file()):
            recovered_v1.add(item['code_sha256'])
hashes -= recovered_v1
print(f'Skipping {len(recovered_v1)} recovered v1/v2 graphs; {len(hashes)} unresolved.', flush=True)
rows_by_hash = {}
manifest = json.loads((ROOT / 'data/manifest.json').read_text())
for dataset in config['datasets']:
    for split, item in manifest['datasets'][dataset].items():
        if not isinstance(item, dict) or 'rows_path' not in item:
            continue
        with (ROOT / item['rows_path']).open() as handle:
            for line in handle:
                row = json.loads(line)
                if row['code_sha256'] in hashes:
                    rows_by_hash.setdefault(row['code_sha256'], row)
missing_rows = hashes - rows_by_hash.keys()
if missing_rows:
    raise RuntimeError(f'Cannot locate source for {len(missing_rows)} empty graphs')
print(f'Retrying {len(hashes)} empty graphs; isolated output: {retry_dir}', flush=True)

# Restore split :: only outside quoted strings and comments.
def restore_scope_tokens(code):
    import re
    token = re.compile(r'R"(?P<delimiter>[^ ()\\\t\r\n]{0,16})\(.*?\)(?P=delimiter)"|"(?:\\.|[^"\\])*"|\'(?:\\.|[^\'\\])*\'|//[^\n]*|/\*.*?\*/|:\s+:', re.S)
    return token.sub(lambda m: '::' if m.group(0).startswith(':') else m.group(0), code)


def project_context(original):
    # Do not expand orphaned token-pasting snippets without their real macro inputs.
    if '##' in original:
        return '', [], [], 'Orphaned token-pasting macro: need original definitions and invocation arguments'
    if re.search(r'(?m)(?<!\S)#\s*(?:if|ifdef|ifndef|else|elif|endif|define)\b', original):
        return '', [], [], 'Preprocessor directives embedded in extracted snippet; need original line structure/build configuration'
    if re.search(r'\b(?:WRITE_CLASS_ENCODER|CONCAT|ARRAY_TYPE|EV_P_|EV_A_)\b', original):
        return '', [], [], 'Semantic/code-generating macro needs original project definitions'
    prefix = ''
    profiles, assumptions = [], []
    php_macros = ('TSRMLS_D', 'TSRMLS_DC', 'TSRMLS_C', 'TSRMLS_CC')
    if any(re.search(r'\b'+name+r'\b', original) for name in php_macros):
        # Actual PHP NTS branch: context macros vanish; NOT valid for an unknown ZTS build.
        prefix += ''.join('#define '+name+'\n' for name in php_macros)
        profiles.append('PHP_NTS')
        assumptions.append('Assumed PHP non-thread-safe build; validate against original build before production use')
    if re.search(r'\bSTACK_OF\s*\(', original):
        prefix += '#define STACK_OF(type) struct stack_st_##type\n'
        profiles.append('OpenSSL_STACK_OF')
    for name, value in [('reg1', 'register'), ('OUT', ''), ('LDAP_CONST', 'const')]:
        if re.search(r'\b'+name+r'\b', original):
            prefix += f'#define {name} {value}\n'
            profiles.append(name)
            assumptions.append(f'Assumed {name} expansion is {value!r}; original header not supplied')
    if not profiles:
        return '', [], [], 'No supported project profile; snippet may be incomplete or not a standalone function'
    return prefix, profiles, assumptions, None

def retry_one(code_hash):
    row = rows_by_hash[code_hash]
    card = retry_dir / code_hash
    card.mkdir(exist_ok=True)
    result_path = card / 'result.json'
    if result_path.exists():
        previous = json.loads(result_path.read_text())
        if previous.get('status') in ('recovered', 'recovered_under_assumption') and (card / 'features.json.gz').is_file():
            return previous
    original = row['code']
    if hashlib.sha256(original.encode('utf-8')).hexdigest() != code_hash:
        raise ValueError(f'Source identity mismatch: {code_hash}')
    # Reject cases needing genuine macro expansion/build context, rather than invent it.
    prefix, extra_profile, assumptions, reason = project_context(original)
    if reason:
        result = {'code_sha256': code_hash, 'record_id': row['record_id'],
                  'status': 'requires_original_context', 'reason': reason,
                  'nodes': 0, 'edges': 0}
        result_path.write_text(json.dumps(result, indent=2))
        return result
    active_macros = [name for name in annotation_macros
                     if re.search(r'\b' + re.escape(name) + r'\b', original)]
    prefix += ''.join(f'#ifndef {name}\n#define {name} {macro_values[name]}\n#endif\n'
                      for name in active_macros)
    parser_input = prefix + restore_scope_tokens(original)
    source = card / 'target.cpp'
    source.write_text(parser_input, encoding='utf-8')
    result = {'code_sha256': code_hash, 'record_id': row['record_id'],
              'parser_input_sha256': hashlib.sha256(parser_input.encode()).hexdigest(),
              'profile_id': profile_id, 'applied_annotations': active_macros, 'project_profile': extra_profile,
              'assumptions': assumptions, 'status': 'failed', 'nodes': 0, 'edges': 0}
    started = time.monotonic()
    try:
        import shutil
        export = card / 'export'
        if export.exists():
            shutil.rmtree(export)
        commands = [
            ('joern-parse', [str(source), '--output', str(card / 'cpg.bin'), '--frontend-args', '--log-problems']),
            ('joern-export', ['--repr', 'all', '--format', 'graphml', '--out', str(export), str(card / 'cpg.bin')])]
        for tool, args in commands:
            completed = subprocess.run([str(JOERN_HOME / tool), '-J-Xms128m', '-J-Xmx2g',
                                        '-J-XX:ActiveProcessorCount=4', *args],
                                       capture_output=True, text=True, errors='replace', timeout=300)
            (card / (tool + '.log')).write_text(completed.stdout + '\n' + completed.stderr)
            if completed.returncode:
                raise RuntimeError(f'{tool} exited {completed.returncode}; see {card / (tool + ".log")}')
        files = [p for p in export.rglob('*') if p.suffix.lower() in ('.xml', '.graphml')]
        if len(files) != 1:
            raise RuntimeError(f'Expected 1 GraphML, found {len(files)}')
        graph = parse_graphml(files[0])
        result.update(nodes=len(graph['nodes']), edges=len(graph['edges']))
        result['status'] = ('recovered_under_assumption' if assumptions else 'recovered') if graph['nodes'] and graph['edges'] else 'still_empty'
        graph.update(schema='grace_h1_graph_v1', code_sha256=code_hash,
                     source='joern_retry_derived_input_v3', retry_provenance=dict(result))
        graph['feature_sha256'] = fingerprint(graph)
        with gzip.open(card / 'features.tmp.gz', 'wt', encoding='utf-8') as handle:
            json.dump(graph, handle, ensure_ascii=False, sort_keys=True)
        os.replace(card / 'features.tmp.gz', card / 'features.json.gz')
    except Exception as exc:
        result['error'] = str(exc)
    result['elapsed_seconds'] = time.monotonic() - started
    temp = card / 'result.tmp.json'
    temp.write_text(json.dumps(result, ensure_ascii=False, indent=2))
    temp.replace(result_path)
    return result

results = []
started = time.monotonic()
workers = min(4, int(config.get('graph_workers', 4)))
with cf.ThreadPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(retry_one, h) for h in sorted(hashes)]
    with tqdm(total=len(futures), desc='Retry empty graphs', unit='graph', mininterval=1) as bar:
        for future in cf.as_completed(futures):
            results.append(future.result())
            from collections import Counter
            counts = Counter(item['status'] for item in results)
            bar.set_postfix(dict(counts), refresh=False)
            bar.update(1)
results.sort(key=lambda item: item['code_sha256'])
report = {'profile': retry_profile, 'profile_id': profile_id, 'total': len(hashes),
          'counts': dict(Counter(item['status'] for item in results)),
          'elapsed_seconds': time.monotonic() - started, 'results': results,
          'production_cache_updated': False, 'prompts_regenerated': False,
          'v1_v2_recovered_preserved': len(recovered_v1)}
report_path = retry_dir / 'report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps({k: v for k, v in report.items() if k not in ('results', 'profile')}, indent=2))
print('Report:', report_path)
print('Graphs remain isolated. recovered_under_assumption requires build-context validation; existing prompts are unchanged.')


Skipping 677 recovered v1/v2 graphs; 128 unresolved.
Retrying 128 empty graphs; isolated output: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/runs/graph_retry_v3/2fb2ab9ea6f09b2f


Retry empty graphs: 100%|██████████| 128/128 [02:12<00:00,  1.04s/graph, requires_original_context=83, recovered_under_assumption=43, recovered=2]

{
  "profile_id": "2fb2ab9ea6f09b2f",
  "total": 128,
  "counts": {
    "requires_original_context": 83,
    "recovered_under_assumption": 43,
    "recovered": 2
  },
  "elapsed_seconds": 132.9126008580006,
  "production_cache_updated": false,
  "prompts_regenerated": false,
  "v1_v2_recovered_preserved": 677
}
Report: /home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade/runs/graph_retry_v3/2fb2ab9ea6f09b2f/report.json
Graphs remain isolated. recovered_under_assumption requires build-context validation; existing prompts are unchanged.
